# Step A — Single-node Denmark: optimal capacity mix

**Task (Assignment 1, part a):** Pick one country and calculate the optimal capacities for
renewable and non-renewable generators. Plot the dispatch for a summer and a winter week,
the annual electricity mix, and investigate the technology contributions using duration
curves / capacity factors.

**Model:** Single bus (Denmark), three extendable generators — combined wind, solar PV, CCGT.

**Requires in the working directory:**
- `DK_2015_merged.csv`
- `functions_to_investigate.py` (custom plotting helpers used below as `fti`)


## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa


## Load 2015 Danish data (demand, wind CF, solar CF)

In [ ]:
dataframe_dk = pd.read_csv("DK_2015_merged.csv", index_col=0, sep=",", parse_dates=True)
demand_dk = dataframe_dk["DK_load_actual_entsoe_transparency"]
CF_wind = dataframe_dk["wind_cf_Unnamed: 1"]
CF_solar = dataframe_dk["pv_cf_Unnamed: 1"]
dataframe_dk.head()


## Technology costs (2020 — DEA LCOE calculator)

Capital cost = Investment / lifetime + Fixed O&M. Marginal cost for CCGT comes from fuel
cost divided by efficiency + variable O&M. See Table 1.1 in the report.


In [ ]:
data = {
    "capital_cost": [
        1500000/25 + 60000,   # wind:  120,000 $/MW/year
        800000/25 + 14000,    # solar:  46,000 $/MW/year
        700000/25 + 24000,    # CCGT:   52,000 $/MW/year
    ],
    "marginal_cost": [0.0, 0.0, 9.5 * 3.6 / 0.56 + 2.30]  # CCGT: ~63.4 $/MWh
}

costs = pd.DataFrame(data, index=["wind_combined", "solar", "CCGT"])
costs.head()


## Build the network

In [ ]:
dnk_n = pypsa.Network()
dnk_n.set_snapshots(dataframe_dk.index.values)


In [ ]:
# Single node for Denmark
dnk_n.add("Bus", "Denmark")


In [ ]:
# Carriers (technology labels; used for colouring)
carriers = ["wind_combined", "solar", "CCGT"]
dnk_n.add(
    "Carrier",
    carriers,
    color=["blue", "red", "brown"],
)


In [ ]:
# Demand associated to the Denmark bus
dnk_n.add("Load", "dnk_demand", bus="Denmark", p_set=demand_dk.values)


In [ ]:
max_val = demand_dk.max()
max_time = demand_dk.idxmax()
print("Peak demand:", max_val, "MW at", max_time)


In [ ]:
dnk_n.loads_t.p_set.plot(figsize=(6, 2), ylabel="MW")


## Generators (extendable — capacity will be optimised)

In [ ]:
# wind onshore (combined)
dnk_n.add(
    "Generator",
    "wind_combined",
    bus="Denmark",
    carrier="wind_combined",
    capital_cost=costs.loc["wind_combined", "capital_cost"],
    marginal_cost=costs.loc["wind_combined", "marginal_cost"],
    p_max_pu=CF_wind.values,
    p_nom_extendable=True,
)


In [ ]:
# solar
dnk_n.add(
    "Generator",
    "solar",
    bus="Denmark",
    carrier="solar",
    capital_cost=costs.loc["solar", "capital_cost"],
    marginal_cost=costs.loc["solar", "marginal_cost"],
    p_max_pu=CF_solar.values,
    p_nom_extendable=True,
)


In [ ]:
# CCGT — efficiency 0.58 is applied by PyPSA to translate fuel marginal cost to electrical
dnk_n.add(
    "Generator",
    "CCGT",
    bus="Denmark",
    carrier="CCGT",
    capital_cost=costs.loc["CCGT", "capital_cost"],
    marginal_cost=costs.loc["CCGT", "marginal_cost"],
    efficiency=0.58,
    p_nom_extendable=True,
)


In [ ]:
dnk_n.generators_t.p_max_pu.loc["2015"].plot(figsize=(6, 2), ylabel="CF")


## Optimise (capacity expansion, linear programming)

In [ ]:
dnk_n.optimize(solver_name="highs")


## Results — raw printouts

In [ ]:
# Objective value [USD/year]
print("objective value: ", dnk_n.objective)


In [ ]:
# Installed capacities [MW]
print("in MW \n", dnk_n.generators.p_nom_opt)


In [ ]:
# Hourly production per technology [MW]
hourly_prod = dnk_n.generators_t.p
print(hourly_prod)
# dnk_n.generators_t.p.to_csv("hourly_dnk_production.csv")


In [ ]:
# Annual production per technology [TWh]
print(" in TWh \n", dnk_n.generators_t.p.sum() / 1e6)


In [ ]:
# Hourly income per technology (production × price) [$/h]
hourly_income = dnk_n.generators_t.p.multiply(dnk_n.buses_t.marginal_price.to_numpy())
print(hourly_income)


In [ ]:
# Annual income per technology [$/y]
incomes_y = dnk_n.generators_t.p.multiply(dnk_n.buses_t.marginal_price.to_numpy()).sum()
print(incomes_y)


In [ ]:
# Annual costs per technology (capex + opex) [$/y]
costs_y = dnk_n.statistics.capex().add(dnk_n.statistics.opex(), fill_value=0)
print(costs_y)


In [ ]:
# Hourly energy prices (marginal price at the Denmark bus) [$/MWh]
energy_prices = dnk_n.buses_t.marginal_price


In [ ]:
# Scarcity revenue = revenue − opex. For the marginal technology at the optimum it
# should equal capex (zero-profit condition for extendable generators).
revenue = dnk_n.statistics.revenue(groupby=False)
opex = dnk_n.statistics.opex(groupby=False)

scarsity_revenue = revenue.sub(opex, fill_value=0)
print(scarsity_revenue.loc["Generator"])
print(dnk_n.statistics.capex())


## Plots (via the `functions_to_investigate` helper module)

These wrap matplotlib around the optimised network for dispatch weeks, duration curves,
the annual mix, mismatch analysis, installed capacity and system costs.


In [ ]:
import importlib
import functions_to_investigate as fti


In [ ]:
fti.plot_generation_mix(dnk_n, '2015-07-01', '2015-08-15')


In [ ]:
fti.plot_generation_mix(dnk_n, '2015-01-01', '2015-02-15')


In [ ]:
fti.calculate_system_metrics(dnk_n, '2015-07-01', '2015-08-31')


In [ ]:
fti.plot_prices_and_scarcity(dnk_n, '2015-07-01', '2015-08-15')


In [ ]:
fti.plot_price_duration_curve(dnk_n, '2015-01-01', '2015-12-31')


In [ ]:
fti.plot_energy_production(dnk_n, '2015-07-01', '2015-08-15')


In [ ]:
fti.plot_annual_mix(dnk_n)


In [ ]:
fti.plot_mismatch_analysis(dnk_n, '2015-01-01', '2015-12-31')


In [ ]:
fti.plot_installed_capacity(dnk_n)


In [ ]:
fti.plot_system_costs(dnk_n)
